In [ ]:
!pip install groq

In [ ]:
import os
from groq import Groq
from getpass import getpass

In [ ]:
os.environ['GROQ_API_KEY'] = getpass("GROQ_API_KEY")

In [ ]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


In [ ]:
# Function to call OpenAI
def get_response_v0(prompt):

    completion = client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=[
          {
              "role": "user",
              "content": prompt,
          },
      ],
    )

    return completion.choices[0].message.content

## Zero Shot Classification

In [7]:
complaint = (
    "The delivery guy just left the package at the gate without ringing the bell. "
    "I found it two hours later completely soaked from the rain."
)

# Without Chain of Thought Prompt
basic_prompt = f"""
Classify the following customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Complaint: "{complaint}"

Category:"""

print(get_response_v0(basic_prompt), "\n")

Category: 1. Delivery issue 

The customer is complaining about the way the package was delivered, specifically that it was left unattended and exposed to the rain, rather than being handed over to them directly or left in a secure location. This issue is related to the delivery process, making it a delivery issue. 



## Few Shots Classification

In [8]:
# Few-shot examples
few_shot_prompt = f"""
Classify the following customer complaints into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Examples:

Complaint: "My order was supposed to arrive yesterday, but it hasn’t shown up yet."
Category: Delivery issue

Complaint: "The shoes I received had a tear on the side."
Category: Product quality

Complaint: "The support agent was rude and didn’t help at all."
Category: Customer service

Complaint: "I had trouble applying the discount code during checkout."
Category: Other

Now classify the following complaint:

Complaint: {complaint}
Category:"""

In [9]:
# Complaint to classify
complaint_to_classify = (
    "The delivery guy just left the package at the gate without ringing the bell. "
    "I found it two hours later completely soaked from the rain."
)

In [10]:
print(few_shot_prompt.format({"complaint": complaint_to_classify[0]}))


Classify the following customer complaints into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Examples:

Complaint: "My order was supposed to arrive yesterday, but it hasn’t shown up yet."
Category: Delivery issue

Complaint: "The shoes I received had a tear on the side."
Category: Product quality

Complaint: "The support agent was rude and didn’t help at all."
Category: Customer service

Complaint: "I had trouble applying the discount code during checkout."
Category: Other

Now classify the following complaint:

Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.
Category:


In [11]:
print(get_response_v0(few_shot_prompt.format({"complaint": complaint_to_classify[0]})))

Category: Delivery issue

The complaint is related to the delivery process, specifically the way the package was left at the customer's location, which led to it getting damaged due to rain. This issue is directly related to the delivery service, making it a delivery issue.


## Chain of Thought

In [12]:
# Function to call OpenAI
def get_response_v2(system_prompt, prompt):

    print( "**************** Prompt *****************")
    print(system_prompt + prompt)
    print( "****************************************")

    completion = client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=[
          {"role": "system", "content": system_prompt},
          {
              "role": "user",
              "content": prompt,
          },
      ],
    )

    return completion.choices[0].message.content

In [13]:
system_prompt = """Your job is to classify complaints into one of the following categories:

Delivery issue
Product quality
Customer service
Other

Use chain-of-thought reasoning to guide the classification of the following complaint."""

user_prompt = f"""
Complaint: {complaint}
"""

print("With Chain of Thought:")
print(get_response_v2(system_prompt,
                      user_prompt.format({"complaint":
                                         "The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain."})))

With Chain of Thought:
**************** Prompt *****************
Your job is to classify complaints into one of the following categories:

Delivery issue
Product quality
Customer service
Other

Use chain-of-thought reasoning to guide the classification of the following complaint.
Complaint: The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain.

****************************************
To classify this complaint, let's break it down step by step:

1. The issue is related to the package being left at the gate without the delivery person notifying the customer.
2. The package was left unattended for two hours, which led to it getting soaked in the rain.
3. The customer's concern is with how the package was handled during delivery, specifically the fact that it was not delivered to their doorstep and they were not notified.

Considering these points, the complaint is primarily about the delivery process and 

## Structured Response

In [ ]:
# System prompt
system_prompt = "You are a helpful assistant that classifies customer complaints. Return the result only as a JSON object: {\"category\": \"...\"}"

In [ ]:
user_prompt = f"""
Classify the customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Complaint: {complaint}
"""

In [ ]:
resp = get_response_v2(system_prompt,
                      user_prompt.format({"complaint":
                                         "The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain."}))

In [ ]:
resp

In [20]:
import json

category_dict = json.loads(resp)

print(category_dict)
print(category_dict["category"])

{'category': 'Delivery issue'}
Delivery issue


## Temperature

In [15]:
# API wrapper
def get_response_v3(system_prompt, prompt, temperature=0.7):

    print( "**************** Prompt *****************")
    print(system_prompt + prompt)
    print( "****************************************")

    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature
    )
    return completion.choices[0].message.content.strip()

In [16]:
system_prompt = """Your job is to classify complaints into one of the following categories:

Delivery issue
Product quality
Customer service
Other

Use chain-of-thought reasoning to guide the classification of the following complaint."""

user_prompt = f"""
Complaint: {complaint}
"""

In [17]:
print(get_response_v3(system_prompt,
                      user_prompt.format({"complaint":
                                         "The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain."})))

**************** Prompt *****************
Your job is to classify complaints into one of the following categories:

Delivery issue
Product quality
Customer service
Other

Use chain-of-thought reasoning to guide the classification of the following complaint.
Complaint: I waited all day at home, but the package never came. When I contacted support,  they told me to wait another 48 hours without giving a reason.

****************************************
To classify this complaint, let's break it down into its components:

1. The customer waited all day for a package that never arrived. This suggests an issue with the delivery of the package.
2. The customer contacted support, which indicates they were seeking help with their issue.
3. The support team's response was to wait another 48 hours without providing a reason, which is related to the customer's interaction with the company's support team.

Considering these points, we can see that the primary issue is with the delivery of the pack

In [18]:
print(get_response_v3(system_prompt,
                      user_prompt.format({"complaint":
                                         "The delivery guy just left the package at the gate without ringing the bell. I found it two hours later completely soaked from the rain."}),
                      temperature = 0.0))

**************** Prompt *****************
Your job is to classify complaints into one of the following categories:

Delivery issue
Product quality
Customer service
Other

Use chain-of-thought reasoning to guide the classification of the following complaint.
Complaint: I waited all day at home, but the package never came. When I contacted support,  they told me to wait another 48 hours without giving a reason.

****************************************
To classify this complaint, let's break it down into its components:

1. The customer waited all day at home for a package, but it never arrived. This suggests that there was an issue with the delivery of the package.

2. The customer then contacted support, which indicates that they were seeking help or resolution for the issue they were experiencing.

3. The support team's response was to tell the customer to wait another 48 hours without providing a reason. This response from support can be seen as unhelpful or unsatisfactory.

Consider

## Self Consistency

In [21]:
import time
from collections import Counter

system_prompt = "You are a helpful assistant that classifies customer complaints. Return the result only as a JSON object: {\"category\": \"...\"}"

user_prompt = f"""
Classify the customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Use chain-of-thought reasoning to guide the classification of the following complaint.

Complaint: {complaint}
"""

complaint = "I waited all day at home, but the package never came. When I contacted support,  they told me to wait another 48 hours without giving a reason."

print(user_prompt.format({"complaint": complaint}))

results = []

for temp in [0.0, 0.5, 2.0]:
    try:
        resp = get_response_v3(system_prompt
                              , user_prompt.format({"complaint": complaint})
                              , temperature = temp)
        category_dict = json.loads(resp)
        category = category_dict['category']
        if category is not None:
            results.append(category)
    except Exception as e:
        print(f"Error on run {temp}: {e}")

    time.sleep(1)  # to avoid rate limits

    counts = Counter(results)
    majority = counts.most_common(1)[0]

print("All predictions:", results)
print("Majority vote:", majority)


Classify the customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Use chain-of-thought reasoning to guide the classification of the following complaint.

Complaint: I waited all day at home, but the package never came. When I contacted support,  they told me to wait another 48 hours without giving a reason.

**************** Prompt *****************
You are a helpful assistant that classifies customer complaints. Return the result only as a JSON object: {"category": "..."}
Classify the customer complaint into one of these categories:
1. Delivery issue
2. Product quality
3. Customer service
4. Other

Use chain-of-thought reasoning to guide the classification of the following complaint.

Complaint: I waited all day at home, but the package never came. When I contacted support,  they told me to wait another 48 hours without giving a reason.

****************************************
**************** Prompt *****************
Yo